# Calculate the portfolio vol

In [1]:
# functions
import yfinance as yf
import pandas as pd
import numpy as np
from __future__ import annotations


class Securities(object):
    def __init__(self, symbol: str = None, securityType: str = None):
        self._symbol = symbol
        self._securityType = securityType

    @property
    def yahoo_symbol(self):
        return self._symbol

    def __eq__(self, security: Securities):
        return (self._symbol == security._symbol) and (self._securityType == security._securityType)

    def __hash__(self):
        return hash(self._symbol + self._securityType)

    def __repr__(self):
        return self._symbol


class Portfolio(object):
    def __init__(self):
        self._inventory = {}

    def add_pos(self, security: Securities = None, quantity: float = None):
        assert security is not None
        assert quantity is not None

        self._inventory[security] = quantity

    def _get_close_price_hist(self, yahoo_symbol, period: str = "30d", interval: str = "1d"):
        stock_data = yf.Ticker(yahoo_symbol)
        hist = stock_data.history(period=period, interval=interval)
        hist = hist[["Close"]]
        hist.columns = [yahoo_symbol]
        return hist

    def get_cov_matrix(self, look_back_days: int = 30):
        self.hist_data = pd.concat(
            [
                self._get_close_price_hist(key.yahoo_symbol, period="%dd" % (look_back_days), interval="1d")
                for key in self._inventory.keys()
            ],
            axis=1,
        )

        self.return_hist = (self.hist_data.diff().shift(periods=-1) / self.hist_data) * 100
        self.cov_data = self.return_hist.dropna().cov()
        return self.cov_data

    def get_corr_matrix(self, look_back_days: int = 30):
        if hasattr(self, "return_hist"):
            return self.return_hist.dropna().corr()
        else:
            self.get_corr_matrix(look_back_days=look_back_days)
            return self.return_hist.dropna().corr()

    def get_portfolio_vol(self, look_back_days: int = 30):
        cov_df = self.get_cov_matrix(look_back_days=look_back_days)
        market_value_df = self.get_portfolio_market_value()
        X = market_value_df[["marketValue"]]
        portfolio_vol = np.sqrt(cov_df.dot(X).T.dot(X)) * np.sqrt(260) / 100.0

        return portfolio_vol

    def _get_latest_price(self, yahoo_symbol: str):
        stock_data = yf.Ticker(yahoo_symbol)
        return stock_data.info["regularMarketPrice"]

    def get_portfolio_market_value(self):
        df = pd.DataFrame(
            {
                key.yahoo_symbol: [
                    self._get_latest_price(key.yahoo_symbol),
                    self._inventory[key],
                ]
                for key in self._inventory.keys()
            },
            index=["price", "quantity"],
        )
        df = df.T
        df["marketValue"] = df["quantity"] * df["price"]
        return df

    def get_portfolio_stats(self, look_back_days: int = 30):
        self.get_cov_matrix(look_back_days=look_back_days)
        df = self.return_hist
        leg_stats = pd.DataFrame(
            {
                "mean": df.mean(),
                "std": df.std(),
                "skew": df.skew(),
                "curtosis": df.kurtosis(),
            }
        )

        market_value = self.get_portfolio_market_value()
        market_value["weight"] = market_value["marketValue"] / market_value["marketValue"].sum()

        port_return_df = df.dot(market_value[["weight"]])
        portfolio_stats = pd.DataFrame(
            {
                "mean": port_return_df.mean(),
                "std": port_return_df.std(),
                "skew": port_return_df.skew(),
                "curtosis": port_return_df.kurtosis(),
            }
        )
        leg_stats["weight"] = market_value["weight"]
        portfolio_stats.index = ["total"]
        portfolio_stats["weight"] = 1
        stats = pd.concat([leg_stats, portfolio_stats])

        return stats


# functions
from scipy import optimize


class SharpeOptimization(object):
    def __init__(self, ret: np.ndarray = None, cov_mat: np.ndarray = None):
        self.ret = np.asarray(ret)
        self.cov_mat = np.asarray(cov_mat)
        self.N = len(self.cov_mat)

    def get_obj_fn(self):
        return lambda x: -self.calc_sharpe(x)

        # def sharpe(x):
        #    y = self.ret.dot(x)
        #    sigma = np.sqrt(self.cov_mat.dot(x).dot(x))
        #    return -y / sigma
        # return sharpe

    def get_bounds(self, bounds: dict = None):
        if bounds is not None:
            lower_bounds = bounds.get("lower_bound", [-np.inf] * N)
            upper_bounds = bounds.get("upper_bound", [np.inf] * N)
        else:
            lower_bounds = [0] * self.N
            upper_bounds = [np.inf] * self.N

        opt_bounds = optimize.Bounds(lower_bounds, upper_bounds)

        return opt_bounds

    def get_constraints(self, linear_constraints: dict = None):
        if linear_constraints is not None:
            mat = linear_constraints["mat"]
            lower_bound = linear_constraints["lower_bound"]
            upper_bound = linear_constraints["upper_bound"]
        else:
            mat = np.ones((1, self.N))
            lower_bound = [1]
            upper_bound = [1]

        opt_constraints = optimize.LinearConstraint(mat, lower_bound, upper_bound)
        return opt_constraints

    def optimize(self, x0: list, bounds: dict = None, linear_constraints: dict = None):
        """
        bound: {lower_bound:[], upper_bound:[]}
        linear_constraints: {mat:[], lower_bound : [], upper_bound: []}
        """

        opt_bounds = self.get_bounds(bounds=bounds)
        opt_constraints = self.get_constraints(linear_constraints=linear_constraints)
        fn = self.get_obj_fn()
        result = optimize.minimize(fn, x0, bounds=opt_bounds, constraints=opt_constraints)
        return result

    def calc_sharpe(self, x):
        x = np.asarray(x)
        y = self.ret.dot(x)
        sigma = np.sqrt(self.cov_mat.dot(x).dot(x))
        return y / sigma

    def calc_port_stats(self, x):
        x = np.asarray(x)
        y = self.ret.dot(x)
        sigma = np.sqrt(self.cov_mat.dot(x).dot(x))
        sharpe = y / sigma
        return {"ret": y, "vol": sigma, "Sharpe": sharpe}

In [2]:
tlt = Securities(symbol="TLT", securityType="ETF")
msft = Securities(symbol="MSFT", securityType="STOCK")
goog = Securities(symbol="GOOGL", securityType="STOCK")
schw = Securities(symbol="SCHW", securityType="STOCK")
xlf = Securities(symbol="XLF", securityType="STOCK")
iglb = Securities(symbol="IGLB", securityType="ETF")
ccl = Securities(symbol="CCL", securityType="STOCK")
look_back_days = int(365 * 3)
total_portfolio_value = 310_000
portfolio = Portfolio()
portfolio.add_pos(tlt, 1100)
portfolio.add_pos(msft, 40.0836)
portfolio.add_pos(schw, 600)
# portfolio.add_pos(iglb, 200)
# portfolio.add_pos(ccl, 500)
# portfolio.add_pos(xlf, 300*2)

In [3]:
cov_data = portfolio.get_cov_matrix(look_back_days=look_back_days)
corr_data = portfolio.get_corr_matrix(look_back_days=look_back_days)

In [4]:
# average daily vol
cov_data

,TLT,MSFT,SCHW
TLT,1.211351,-0.318121,-1.036201
MSFT,-0.318121,4.026095,2.017688
SCHW,-1.036201,2.017688,6.350166


In [5]:
corr_data.round(3)

,TLT,MSFT,SCHW
TLT,1.000,-0.144,-0.374
MSFT,-0.144,1.000,0.399
SCHW,-0.374,0.399,1.000


In [6]:
portfolio.get_portfolio_market_value()

,price,quantity,marketValue
TLT,105.82,1100.0000,116402.000000
MSFT,295.37,40.0836,11839.492932
SCHW,50.60,600.0000,30360.000000


In [7]:
year_vol = portfolio.get_portfolio_vol(look_back_days=look_back_days)
daily_vol = year_vol / np.sqrt(252)
print("year vol %.1f, daily vol %.1f" % (year_vol.iloc[0, 0], daily_vol.iloc[0, 0]))
print(
    "year vol %.1f%%, daily vol %.1f%%"
    % (
        100 * year_vol.iloc[0, 0] / total_portfolio_value,
        100 * daily_vol.iloc[0, 0] / total_portfolio_value,
    )
)

year vol 20447.3, daily vol 1288.1
year vol 6.6%, daily vol 0.4%


In [8]:
portfolio.get_portfolio_stats(look_back_days=look_back_days)

,mean,std,skew,curtosis,weight
TLT,0.001343,1.100615,0.258169,5.479545,0.733928
MSFT,0.120005,2.006513,0.028537,6.764097,0.074649
SCHW,0.059267,2.519954,0.087095,6.427818,0.191423
total,0.021289,0.799545,-0.132991,3.945854,1.000000


# Min Var Optimization

In [9]:
# import packages
import sys

sys.path.append("../src/")

In [10]:
from systrading.sysportopt.portopt import NormalizedMinVolOptimization

In [11]:
# configuration of returns
exp_anual_return_config = pd.Series({tlt: 0.20, msft: 0.10, schw: 0.30})
total_risk = 310_000 * 0.10
year_days = 252
risk_penalty = 3
sharpe_cap = 2.0

In [12]:
# setup optimizer
min_var_opt = NormalizedMinVolOptimization(risk_penalty=risk_penalty, sharpe_cap=sharpe_cap)
cov_mat = portfolio.get_cov_matrix(look_back_days=look_back_days)
cov_mat_yearly = cov_mat * year_days / 1e4  # divided by 1e4 to scale 100 percent

min_var_opt.set_cov_mat(cov_mat_yearly)
min_var_opt.set_return(exp_anual_return_config)
min_var_opt.lower_bounds = [0] * min_var_opt.N

result = min_var_opt.port_optimize()

In [13]:
result

{'portRet': 0.23203944129456622,
 'portVol': 0.16677322068040382,
 'portSharpe': 1.3913471260427084,
 'weights': array([0.49837861, 0.09061349, 0.4110079 ]),
 'success': True}

In [14]:
pd.DataFrame(
    min_var_opt.corr_mat.round(2),
    index=exp_anual_return_config.keys(),
    columns=exp_anual_return_config.keys(),
)

,TLT,MSFT,SCHW
TLT,1.00,-0.14,-0.37
MSFT,-0.14,1.00,0.40
SCHW,-0.37,0.40,1.00


In [15]:
captial_allocation = min_var_opt.portfolio_capital_allocation(result["weights"], total_risk)

In [16]:
pd.Series(captial_allocation.round(0), index=exp_anual_return_config.keys())

TLT     88427.0
MSFT     8819.0
SCHW    31851.0
dtype: float64

In [17]:
1 / np.sqrt(2 / np.pi) * np.sqrt(252)

19.895745131869624